# Demo 3: Contradiction Detection via Ninai Agents

## The Real-World Problem

**Scenario:** An incident happens. Multiple teams investigate independently:

| Team | Report |
|------|--------|
| Infrastructure | "Root cause: DNS misconfiguration" |
| Database | "DNS is fine, PostgreSQL connection saturation" |
| Ops | "Incident resolved and closed" |
| Support | "Customers still reporting failures" |

**The Challenge:**
- ❌ Without Ninai: You manually read all reports, spend hours deciding which team is correct, miss that there are 2 contradictions, and only 1 team might know the real answer
- ✅ With Ninai: One API call returns who's right, why, and what's actually happening

**What Ninai Does (in seconds):**
1. Identifies contradictions: DNS vs PostgreSQL vs resolution status
2. Scores credibility: Which team's reports are more trustworthy?
3. Causal analysis: Did the DNS fix actually resolve customer failures?
4. Anomaly detection: Status says "closed" but customers STILL failing = 🚩
5. Recommends: "Investigate PostgreSQL despite DNS fix"
6. Shows reasoning: Here's HOW we reached this conclusion

---

## Learning Goals

1. **Understand the problem** — Organizational contradictions waste hours
2. **Create contradictory memories** (same as Demo 1 & 2)
3. **Use Cognitive Gateway** — Delegate analysis to backend agents
4. **See intelligent results** — Contradiction detection with reasoning chains
5. **Realize the shift** — Ninai thinks, you just ask questions

## What You'll Learn

- ✓ Write contradictory memories to Ninai
- ✓ Call Ninai's backend agents through SDK (via Cognitive Gateway)
- ✓ Receive intelligent analysis: what contradicts, credibility scores, causal links
- ✓ See reasoning chains showing HOW Ninai decided
- ✓ Get actionable recommendations with confidence

**Time to insight: Seconds, not hours.** 🚀

## Step 1: Setup and Login

Run the next cell to initialize the SDK and authenticate.

Expected outcome:
- No errors.
- A `client` object is ready.
- You are authenticated with the demo account.
- A unique `seed` value is created to isolate this run from older notebook data.

In [ ]:
from ninai import NinaiClient
import uuid
from datetime import datetime, timezone, timedelta

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

## Step 2: Create Contradictory Memories

**⚠️ Important:** Do NOT re-run Step 1 after this point. The seed will change and Step 3 won't find these memories.

Run the next cell to:
1. Create 4 memories with intentional contradictions about an incident.
2. Each memory represents a different team's assessment with conflicting information.
3. Tag them all with the same `seed` so we can search them together.

Expected outcome:
- 4 memories created successfully.
- All tagged with the same `seed` value.
- Memory content includes deliberate conflicts: DNS claims, PostgreSQL claims, and resolution status conflicts.

In [ ]:
# Create memories with intentional contradictions
print("=" * 80)
print("STEP 2: Creating Contradictory Memories")
print("=" * 80)
print(f"\nIMPORTANT: Seed ID = {seed}")
print("   Step 3 will search for this exact seed.")
print("   DO NOT re-run Step 1 or the seed will change!\n")

entries = [
    f'Infrastructure: outage caused by DNS misconfiguration seed={seed}',
    f'Database team: DNS is healthy, the root cause is PostgreSQL connection saturation seed={seed}',
    f'Ops report: incident resolved and closed seed={seed}',
    f'Support notes: incident resolved but customers still reporting failures seed={seed}',
]

base_time_utc = datetime.now(timezone.utc)
created_ids = []
for idx, entry in enumerate(entries, 1):
    mem = client.memories.create(
        content=entry,
        source_type='manual',
        tags=['incident', 'outage', seed],
        occurred_at=base_time_utc + timedelta(minutes=idx - 1)
    )
    created_ids.append(mem.id)
    print(f'Created memory {idx}/{len(entries)}: {mem.id}')

print(f'\nTotal memories created: {len(created_ids)}')
print(f'Seed: {seed}')
print(f'UTC start timestamp: {base_time_utc.isoformat()}')
print(f'\nProceed to Step 3')
print(f'IMPORTANT: Do NOT re-run Step 1 or the seed will change!')

STEP 2: Creating Contradictory Memories

IMPORTANT: Seed ID = 27a6231f
   Step 3 will search for this exact seed.
   DO NOT re-run Step 1 or the seed will change!

Created memory 1/4: 4bd05a5a-44cf-4191-96e4-80fabd47c3c4
Created memory 2/4: 54957527-5449-4c07-a024-e942df74a91f
Created memory 3/4: 4f3539bf-898f-4ca4-b5c6-dbdee0024098
Created memory 4/4: faa0c065-ebd7-4930-a889-cc869988363f

Total memories created: 4
Seed: 27a6231f

Proceed to Step 3
IMPORTANT: Do NOT re-run Step 1 or the seed will change!


## Step 3: Search and Detect Contradictions

Run the next cell to:
1. Search for all memories matching the current `seed`.
2. Define a robust contradiction detector that identifies conflicting claims.
3. Report which specific contradictions were found and in which memories.

Expected outcome:
- Search returns 4 memories (or close to it).
- The detector identifies at least 2-3 contradictions:
  - Root cause conflict: DNS vs PostgreSQL
  - Status conflict: resolved vs still failing
- Output shows which memory IDs contain each contradiction.

In [3]:
# Step 3: Use Ninai's Cognitive Gateway via SDK
print("=" * 80)
print("STEP 3: Delegating to Ninai's Backend Agents")
print("=" * 80)
print(f"\nAnalyzing memories created with seed: {seed}\n")

# First, retrieve the memories we created
search_results = client.memories.search(query=f'seed={seed}', limit=20, hybrid=True)
print(f"Retrieved {len(search_results.items)} memories:")
for mem in search_results.items:
    print(f"  - {mem.id}: {mem.content_preview[:75]}...")

if not search_results.items:
    print("\nERROR: No memories found. Troubleshooting:")
    print("  1. Verify Step 2 ran successfully")
    print("  2. Ensure you did NOT re-run Step 1 (changes the seed)")
    print("  3. Wait 2-3 seconds for search indexing")
else:
    print("\n" + "=" * 80)
    print("INVOKING COGNITIVE GATEWAY (via SDK)")
    print("=" * 80)

    # Combine memory content
    combined_content = "\n\n".join([
        f"[Team {i}] {mem.content_preview}"
        for i, mem in enumerate(search_results.items, 1)
    ])

    try:
        print("\nCalling client.cognitive.gateway.decide()...")
        print("  -> ConflictDetectionAgent analyzes for contradictions")
        print("  -> Multi-agent reasoning evaluates credibility")
        print("  -> You get back: decision + reasoning chains\n")

        # Call Ninai's backend agents via SDK
        decision_result = client.cognitive.gateway.decide(
            content=combined_content,
            enrichment={
                "analysis_type": "contradiction_detection",
                "context": "Analyzing team reports for conflicting root causes and status mismatches"
            }
        )

        print("Success! Ninai analysis complete.\n")

        # Display results
        print("=" * 80)
        print("NINAI'S ANALYSIS RESULTS")
        print("=" * 80)
        print(f"\nDecision: {decision_result.get('decision', 'N/A')}")
        print(f"Confidence: {decision_result.get('confidence', 0):.2%}")
        print(f"Tone: {decision_result.get('tone', 'neutral')}")

        if decision_result.get('action_recommended'):
            print(f"Recommended Action: {decision_result['action_recommended']}")

        # Show which agents ran
        agents_run = decision_result.get('agents_run', [])
        if agents_run:
            print(f"\nAgents involved ({len(agents_run)}):")
            for agent_name in agents_run:
                print(f"  - {agent_name}")

        # Show reasoning chain
        debate = decision_result.get('debate_transcript', [])
        if debate:
            print(f"\nReasoning Chain ({len(debate)} steps):")
            for idx, step in enumerate(debate, 1):
                if isinstance(step, dict):
                    reasoning = step.get('reasoning', str(step))[:80]
                    agent = step.get('agent', 'Agent')
                    print(f"  {idx}. [{agent}] {reasoning}...")
                else:
                    print(f"  {idx}. {str(step)[:80]}...")

        # Show enrichment
        enrichment = decision_result.get('enrichment', {})
        if enrichment:
            print(f"\nEnrichment:")
            for key, value in enrichment.items():
                if isinstance(value, (str, int, float, bool)):
                    print(f"  - {key}: {value}")

    except AttributeError as e:
        print(f"ERROR: SDK method not available: {e}")
        print("\nFix: Update SDK version with cognitive gateway support")
        print("  pip install --upgrade ninai-sdk")
    except Exception as e:
        print(f"ERROR: {e}")
        print("\nTroubleshooting:")
        print("  - Backend running? cd repos/ninai/backend && uvicorn app.main:app")
        print("  - /cognitive/gateway/decide endpoint exists?")
        print("  - Valid auth token?")

print("\n" + "=" * 80)
print("KEY INSIGHT")
print("=" * 80)
print("""
SDK Integration Pattern:
  1. client.memories.create() → Store data
  2. client.memories.search() → Retrieve data  
  3. client.cognitive.gateway.decide() → Ninai analyzes via agents
  4. Returns: decision + agents_run + debate_transcript

This is how you leverage Ninai's intelligence!
""")

STEP 3: Delegating to Ninai's Backend Agents

Analyzing memories created with seed: 27a6231f

Retrieved 4 memories:
  - 4bd05a5a-44cf-4191-96e4-80fabd47c3c4: Infrastructure: outage caused by DNS misconfiguration seed=27a6231f...
  - 4f3539bf-898f-4ca4-b5c6-dbdee0024098: Ops report: incident resolved and closed seed=27a6231f...
  - faa0c065-ebd7-4930-a889-cc869988363f: Support notes: incident resolved but customers still reporting failures see...
  - 54957527-5449-4c07-a024-e942df74a91f: Database team: DNS is healthy, the root cause is PostgreSQL connection satu...

INVOKING COGNITIVE GATEWAY (via SDK)

Calling client.cognitive.gateway.decide()...
  -> ConflictDetectionAgent analyzes for contradictions
  -> Multi-agent reasoning evaluates credibility
  -> You get back: decision + reasoning chains

Success! Ninai analysis complete.

NINAI'S ANALYSIS RESULTS

Decision: escalate
Confidence: 75.00%
Tone: informational
Recommended Action: Escalate to engineering lead.

Agents involved (4):
 

## Step 4: Understanding Ninai's Intelligence Architecture

### What Just Happened

You didn't write pattern matching code. Instead:

1. **You provided raw data** (contradictory memories)
2. **You asked Ninai a question** via Cognitive Gateway
3. **Ninai's backend agents analyzed it:**
   - ConflictDetectionAgent (Phase 13)
   - CredibilityAgent (Phase 19)
   - CausalReasoningAgent (Phase 12)
   - AnomalyDetectionAgent (Phase 25)
   - ...and multi-agent orchestration
4. **You received intelligent analysis** with reasoning chains

### The Cognitive Gateway Architecture

```
┌─────────────────┐
│ You (SDK User)  │
├─────────────────┤
│ Step 1: Write   │ client.memories.create()
│ Step 2: Read    │ client.memories.search()
│ Step 3: Decide  │ client.cognitive.gateway.decide()  ← We are here
└────────┬────────┘
         │
    (REST API call)
         │
         ↓
┌──────────────────────────────────────────┐
│ Ninai Backend: Cognitive Gateway          │
├──────────────────────────────────────────┤
│ • Input: content + enrichment             │
│ • Delegates to agent orchestration        │
│ • ConflictDetectionAgent runs             │
│ • Multi-agent reasoning happens           │
│ • Credibility scoring                     │
│ ├─ Output: decision + confidence          │
│ ├─ Output: reasoning chains               │
│ ├─ Output: agents_run list                │
│ └─ Output: enrichment insights            │
└──────────────────────────────────────────┘
```

### What Each Agent Does

| Agent | Phase | What It Does |
|-------|-------|-------------|
| ConflictDetectionAgent | 13 | Identifies contradictions between memories |
| CredibilityAgent | 19 | Scores source credibility & trustworthiness |
| CausalReasoningAgent | 12 | Links root causes to effects |
| AnomalyDetectionAgent | 25 | Flags unusual patterns |
| OrchestrationBusAgent | 29 | Coordinates multi-agent reasoning |
| AuditTrailAgent | 31 | Generates explainability chains |

### Expected Results

Run Step 3 again and look for:

- ✓ **decision** — What Ninai concluded
- ✓ **confidence** — How confident (0.0 to 1.0)
- ✓ **agents_run** — Which agents analyzed your data
- ✓ **debate_transcript** — Reasoning steps showing HOW Ninai reached the conclusion
- ✓ **enrichment** — What Ninai learned about your memories

### If Step 3 Fails

If `client.cognitive.gateway.decide()` is not available, check:
1. SDK version — may need latest version
2. Backend running — `/cognitive/gateway/decide` endpoint must exist
3. Org permissions — may need admin role

### This is How Ninai Works

**Ninai is NOT:**
- ✗ Just a database
- ✗ A search engine
- ✗ A place where YOU write analysis code

**Ninai IS:**
- ✓ A cognitive partner
- ✓ Running 80+ intelligent agents
- ✓ Making decisions for you
- ✓ Providing reasoning chains
- ✓ Building organizational memory

### Next Steps

1. **Demo 4** — Role-based briefing (how to surface insights for specific roles)
2. **Cognitive Gateway** — Master all 5 verbs: `write`, `read`, `decide`, `plan`, `explain`
3. **Agent Registry** — See all 80 phases and what each agent does

## Step 5: Troubleshooting — If Ninai's Analysis Didn't Work

If Step 3 threw an error or `client.cognitive.gateway.decide()` is not available, try these checks:

In [4]:
# Troubleshooting: Check backend connection and SDK capability
print("=" * 80)
print("TROUBLESHOOTING: SDK & Backend Status")
print("=" * 80)

# Check 1: SDK has cognitive gateway
print("\n[1] Verifying SDK version has CognitiveGateway...")
try:
    if hasattr(client, 'cognitive') and hasattr(client.cognitive, 'gateway'):
        print("OK: SDK has client.cognitive.gateway")
        print("    (SDK version 0.1.0+ with CognitiveGatewayResource)")
    else:
        print("ERROR: SDK missing cognitive gateway")
        print("  Fix: pip install --upgrade ninai-sdk")
except Exception as e:
    print(f"ERROR: {e}")

# Check 2: Gateway methods available
print("\n[2] Checking gateway methods...")
try:
    gateway = client.cognitive.gateway
    methods = ['write', 'read', 'decide', 'plan', 'explain']
    available = [m for m in methods if hasattr(gateway, m)]
    print(f"OK: Found {len(available)} methods: {', '.join(available)}")
except Exception as e:
    print(f"ERROR: {e}")

# Check 3: Verify memories exist
print("\n[3] Verifying prerequisite memories...")
try:
    verify_search = client.memories.search(query=f'seed={seed}', limit=5)
    if verify_search.items:
        print(f"OK: Found {len(verify_search.items)} memories with seed={seed}")
    else:
        print(f"ERROR: No memories found with seed={seed}")
        print("  Action: Go back to Step 2 and verify memories were created")
except Exception as e:
    print(f"ERROR: Search failed: {e}")

# Check 4: Test backend connectivity
print("\n[4] Testing backend connectivity...")
try:
    # Try a simple gateway.read() call
    test_result = client.cognitive.gateway.read(
        query="test",
        limit=1
    )
    print("OK: Backend /cognitive/gateway/read endpoint is responding")
except Exception as e:
    error_str = str(e)[:80]
    print(f"ERROR: Backend may be offline or endpoint missing: {error_str}")
    print("  Action: Start backend: cd repos/ninai/backend && uvicorn app.main:app --reload")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("""
Demo 3 Checklist:
  [x] SDK version 0.1.0+ installed (has CognitiveGatewayResource)
  [x] client.cognitive.gateway accessible
  [x] All 5 verbs available (write, read, decide, plan, explain)
  [?] Backend running on https://admin.ninai.sansten.com
  [?] /cognitive/gateway/decide endpoint responding

If Step 3 failed:
  1. Verify SDK is version 0.1.0+
  2. Check backend is running
  3. Ensure auth token is valid
  4. Verify org has admin permissions

SDK Response Format:
{
    "decision": "What Ninai concluded",
    "confidence": 0.XX,
    "agents_run": ["ConflictDetectionAgent", "CredibilityAgent", ...],
    "debate_transcript": [steps showing reasoning],
    "enrichment": {additional insights}
}
""")

TROUBLESHOOTING: SDK & Backend Status

[1] Verifying SDK version has CognitiveGateway...
OK: SDK has client.cognitive.gateway
    (SDK version 0.1.0+ with CognitiveGatewayResource)

[2] Checking gateway methods...
OK: Found 5 methods: write, read, decide, plan, explain

[3] Verifying prerequisite memories...
OK: Found 4 memories with seed=27a6231f

[4] Testing backend connectivity...
OK: Backend /cognitive/gateway/read endpoint is responding

SUMMARY

Demo 3 Checklist:
  [x] SDK version 0.1.0+ installed (has CognitiveGatewayResource)
  [x] client.cognitive.gateway accessible
  [x] All 5 verbs available (write, read, decide, plan, explain)
  [?] Backend running on https://admin.ninai.sansten.com
  [?] /cognitive/gateway/decide endpoint responding

If Step 3 failed:
  1. Verify SDK is version 0.1.0+
  2. Check backend is running
  3. Ensure auth token is valid
  4. Verify org has admin permissions

SDK Response Format:
{
    "decision": "What Ninai concluded",
    "confidence": 0.XX,
   